# LiveCodeBench: prompts por nivel + auditoría
Ubicación: `modeling/notebooks/prompts_livecodebench.ipynb`.

Lee los `test*.jsonl` de `data/raw/livecodebench` línea por línea y se queda solo con los campos que sirven (los casos de prueba privados pesan mucho y no se cargan). Los resultados van a `data/processed/por_nivel_livecodebench/` para no mezclarse con los de TACO.

In [9]:
import json
from pathlib import Path
import pandas as pd

DATA = Path("../data")
RAW = DATA / "raw" / "livecodebench"
OUT = DATA / "processed" / "por_nivel_livecodebench"
OUT.mkdir(parents=True, exist_ok=True)

NIVELES = ["easy", "medium", "hard"]

archivos = sorted(RAW.glob("test*.jsonl"))
for a in archivos:
    print(a.name, f"{a.stat().st_size/1e6:.1f} MB")

test.jsonl 1252.6 MB
test2.jsonl 713.4 MB
test3.jsonl 623.4 MB
test4.jsonl 1204.6 MB
test5.jsonl 557.7 MB
test6.jsonl 134.3 MB


## 1. Revisar los campos disponibles
Se lee solo la primera línea de un archivo para ver qué trae.

In [2]:
with open(archivos[0], encoding="utf-8") as f:
    primera = json.loads(f.readline())
for k, v in primera.items():
    print(f"{k:22s} {type(v).__name__:6s} {str(v)[:70]!r}")

question_title         str    'A. Short Sort'
question_content       str    'There are three cards with letters $\\texttt{a}$, $\\texttt{b}$, $\\textt'
platform               str    'codeforces'
question_id            str    '1873_A'
contest_id             str    '1873'
contest_date           str    '2023-08-21T00:00:00'
starter_code           str    ''
difficulty             str    'easy'
public_test_cases      str    '[{"input": "6\\nabc\\nacb\\nbac\\nbca\\ncab\\ncba\\n", "output": "YES\\nYES\\nY'
private_test_cases     str    'eJxrYJmaz8gABhEZQEZ0tVJmXkFpiZKVgpJhTF5iUnJMnpKOglJ+aQlUNNI1GCJUklpcUl'
metadata               str    '{}'


## 2. Cargar solo lo necesario
Si algún nombre de campo es distinto en tu versión, cámbialo en `CAMPOS`.

In [3]:
CAMPOS = ["question_id", "question_title", "question_content", "platform",
          "contest_id", "contest_date", "difficulty", "starter_code"]

filas = []
for a in archivos:
    with open(a, encoding="utf-8") as f:
        for n, linea in enumerate(f):
            linea = linea.strip()
            if not linea:
                continue
            r = json.loads(linea)
            fila = {c: r.get(c) for c in CAMPOS}
            fila["archivo"] = a.name
            fila["linea"] = n
            filas.append(fila)

df = pd.DataFrame(filas).rename(columns={"question_content": "prompt"})
df["prompt"] = df["prompt"].fillna("").astype(str)
df["n_chars"] = df["prompt"].str.len()
df["difficulty"] = pd.Categorical(df["difficulty"].astype(str).str.lower(),
                                  categories=NIVELES, ordered=True)
print(df.shape)
df.head()

(1055, 11)


,question_id,question_title,prompt,platform,contest_id,contest_date,difficulty,starter_code,archivo,linea,n_chars
0,1873_A,A. Short Sort,There are three cards with letters $\texttt{a}...,codeforces,1873,2023-08-21T00:00:00,easy,,test.jsonl,0,1365
1,1873_B,B. Good Kid,Slavic is preparing a present for a friend's b...,codeforces,1873,2023-08-21T00:00:00,easy,,test.jsonl,1,911
2,1873_D,D. 1D Eraser,You are given a strip of paper $s$ that is $n$...,codeforces,1873,2023-08-21T00:00:00,easy,,test.jsonl,2,1537
3,1883_B,B. Chemistry,"You are given a string $s$ of length $n$, cons...",codeforces,1883,2023-09-22T00:00:00,medium,,test.jsonl,3,2465
4,1883_C,C. Raspberries,"You are given an array of integers $a_1, a_2, ...",codeforces,1883,2023-09-22T00:00:00,medium,,test.jsonl,4,1827


## 3. Auditoría

In [4]:
print("Total de problemas:", len(df))
print("Vacíos:", (df["n_chars"] == 0).sum())
print("Muy cortos (<50 caracteres):", (df["n_chars"] < 50).sum())
print("Texto repetido:", df["prompt"].duplicated().sum())
print("question_id repetido:", df["question_id"].duplicated().sum())
print("Sin nivel válido:", df["difficulty"].isna().sum())
print("\nProblemas por archivo:")
print(df["archivo"].value_counts().sort_index())
print("\nProblemas por nivel:")
print(df["difficulty"].value_counts(dropna=False).sort_index())
print("\nLargo del prompt por nivel (caracteres):")
print(df.groupby("difficulty", observed=True)["n_chars"].describe().round(0))

Total de problemas: 1055
Vacíos: 0
Muy cortos (<50 caracteres): 0
Texto repetido: 0
question_id repetido: 0
Sin nivel válido: 0

Problemas por archivo:
archivo
test.jsonl     400
test2.jsonl    111
test3.jsonl    101
test4.jsonl    101
test5.jsonl    167
test6.jsonl    175
Name: count, dtype: int64

Problemas por nivel:
difficulty
easy      322
medium    383
hard      350
Name: count, dtype: int64

Largo del prompt por nivel (caracteres):
            count    mean    std    min     25%     50%     75%     max
difficulty                                                             
easy        322.0  1100.0  423.0  449.0   806.0  1019.0  1324.0  3073.0
medium      383.0  1432.0  550.0  433.0  1054.0  1345.0  1724.0  3823.0
hard        350.0  1666.0  620.0  495.0  1239.0  1569.0  1991.0  3950.0


In [5]:
# Nivel x plataforma
pd.crosstab(df["difficulty"], df["platform"], margins=True)

platform,atcoder,codeforces,leetcode,All
difficulty,,,,
easy,183,4,135,322
medium,174,2,207,383
hard,245,3,102,350
All,602,9,444,1055


In [6]:
# Rango de fechas de los concursos por archivo (sirve para hablar de contaminación de datos)
df["fecha"] = pd.to_datetime(df["contest_date"], errors="coerce")
df.groupby("archivo")["fecha"].agg(["min", "max", "count"])

,min,max,count
archivo,,,
test.jsonl,2023-05-07 00:00:00,2024-03-02 00:00:00,400
test2.jsonl,2024-03-09 00:00:00,2024-05-25 00:00:00,111
test3.jsonl,2024-06-01 00:00:00,2024-08-10 00:00:00,101
test4.jsonl,2023-08-26 19:30:00,2024-10-05 19:30:00,101
test5.jsonl,2024-09-22 00:00:00,2025-01-04 18:30:00,167
test6.jsonl,2025-01-04 00:00:00,2025-04-06 00:00:00,175


In [7]:
# Casos para revisar a mano
raros = df[(df["n_chars"] < 50) | df["prompt"].duplicated(keep=False) | df["question_id"].duplicated(keep=False)]
raros[["archivo", "question_id", "platform", "difficulty", "n_chars", "prompt"]].head(30)

,archivo,question_id,platform,difficulty,n_chars,prompt


## 4. Guardar los prompts por nivel

In [8]:
base = df[(df["n_chars"] >= 50) & df["difficulty"].notna()]
base = base.drop_duplicates("question_id").drop_duplicates("prompt")

for nivel, g in base.groupby("difficulty", observed=True):
    if len(g) == 0:
        continue
    ruta = OUT / f"{nivel}.jsonl"
    g[["question_id", "question_title", "platform", "contest_date", "difficulty", "prompt", "starter_code"]] \
        .to_json(ruta, orient="records", lines=True, force_ascii=False)
    print(f"{nivel:8s} {len(g):5d} prompts -> {ruta}")

print("\nDescartados (vacíos, cortos, repetidos o sin nivel):", len(df) - len(base))

easy       322 prompts -> ..\data\processed\por_nivel_livecodebench\easy.jsonl
medium     383 prompts -> ..\data\processed\por_nivel_livecodebench\medium.jsonl
hard       350 prompts -> ..\data\processed\por_nivel_livecodebench\hard.jsonl

Descartados (vacíos, cortos, repetidos o sin nivel): 0
